In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WaterQualitySparkProject") \
    .getOrCreate()

In [5]:
df = spark.read.csv(
    r"C:\Water_Monitoring_System\data\brisbane_water_quality.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+----------------------------------------+-----+------------+--------+------------------+--------------------+------------------------------+---------+-------------------+
|          Timestamp|Record number|Average Water Speed|Average Water Direction|Chlorophyll|Chlorophyll [quality]|Temperature|Temperature [quality]|Dissolved Oxygen|Dissolved Oxygen [quality]|Dissolved Oxygen (%Saturation)|Dissolved Oxygen (%Saturation) [quality]|   pH|pH [quality]|Salinity|Salinity [quality]|Specific Conductance|Specific Conductance [quality]|Turbidity|Turbidity [quality]|
+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+-----------------------

In [9]:
# Spark job execution
print("Total rows:", df.count())

df.select("Temperature", "pH").show(10)

Total rows: 30894
+-----------+-----+
|Temperature|   pH|
+-----------+-----+
|     20.018|8.176|
|     19.986|8.175|
|     20.001|8.171|
|     19.983|8.171|
|     19.986|8.171|
|     19.834|8.158|
|     19.829|8.158|
|     19.822|8.159|
|     19.804|8.166|
|      19.77|8.168|
+-----------+-----+
only showing top 10 rows


In [15]:
# Transformations
df2 = df.filter(df["Temperature"] > 20)
df3 = df2.select("Timestamp", "Temperature", "pH")
# Actions 
df3.show(10)
df3.count()

+-------------------+-----------+-----+
|          Timestamp|Temperature|   pH|
+-------------------+-----------+-----+
|2023-08-04 23:00:00|     20.018|8.176|
|2023-08-04 23:00:00|     20.001|8.171|
|2023-08-05 12:30:00|     20.007|8.158|
|2023-08-05 13:00:00|      20.21|8.162|
|2023-08-05 13:30:00|     20.368|8.164|
|2023-08-05 14:00:00|     20.415|8.163|
|2023-08-05 15:00:00|     20.463|8.167|
|2023-08-05 15:30:00|     20.391|8.169|
|2023-08-05 16:00:00|      20.37|8.171|
|2023-08-05 16:30:00|     20.428|8.171|
+-------------------+-----------+-----+
only showing top 10 rows


21980

In [17]:
from pyspark.sql.functions import avg

df.groupBy().agg(avg("Temperature")).show()

+------------------+
|  avg(Temperature)|
+------------------+
|24.415394247959632|
+------------------+



In [21]:
# Spark SQL
df.createOrReplaceTempView("water")

In [23]:
spark.sql("""
SELECT AVG(Temperature) as avg_temp
FROM water
""").show()

+------------------+
|          avg_temp|
+------------------+
|24.415394247959632|
+------------------+



In [25]:
spark.sql("""
SELECT Timestamp, pH, Temperature
FROM water
WHERE pH > 8
""").show()

+-------------------+-----+-----------+
|          Timestamp|   pH|Temperature|
+-------------------+-----+-----------+
|2023-08-04 23:00:00|8.176|     20.018|
|2023-08-04 23:30:00|8.175|     19.986|
|2023-08-04 23:00:00|8.171|     20.001|
|2023-08-04 23:30:00|8.171|     19.983|
|2023-08-04 23:00:00|8.171|     19.986|
|2023-08-04 23:00:00|8.158|     19.834|
|2023-08-04 23:30:00|8.158|     19.829|
|2023-08-05 00:00:00|8.159|     19.822|
|2023-08-05 00:30:00|8.166|     19.804|
|2023-08-05 01:00:00|8.168|      19.77|
|2023-08-05 01:30:00|8.167|     19.741|
|2023-08-05 02:00:00| 8.17|     19.712|
|2023-08-05 02:30:00|8.169|     19.711|
|2023-08-05 03:00:00| 8.17|     19.703|
|2023-08-05 03:30:00|8.174|     19.701|
|2023-08-05 04:00:00|8.166|     19.652|
|2023-08-05 04:30:00|8.168|     19.634|
|2023-08-05 05:00:00|8.169|      19.59|
|2023-08-05 05:30:00|8.168|     19.585|
|2023-08-05 06:00:00|8.161|     19.555|
+-------------------+-----+-----------+
only showing top 20 rows


In [27]:
# PySpark
# Partitioning
df_partitioned = df.repartition(4)
print(df_partitioned.rdd.getNumPartitions())

4


In [29]:
# Caching 
df.cache()
df.count()

30894